# 06 - Model Evaluation (Forecast Engine)

## Objective
Evaluate the trained ML pipeline used by StockMetrics to predict **next-day returns**.
This notebook verifies whether the model meets the **ML Business Case** success criteria
and produces evaluation artefacts for the Streamlit dashboard and project documentation.

## Inputs
- Feature dataset: `data/processed/<version>/features_<version>_latest.csv`
- Trained model pipeline: `models/stock_forecast_model_<version>.pkl`

## Outputs
Saved to `outputs/<version>/`:
- Metrics report JSON: `reports/model_evaluation_report_<version>.json`
- Predictions CSVs: `reports/predictions_train_<version>.csv`, `reports/predictions_test_<version>.csv`
- Plots:
  - `figures/eval_actual_vs_pred_train_<version>.png`
  - `figures/eval_actual_vs_pred_test_<version>.png`
  - `figures/eval_residuals_hist_test_<version>.png`
  - `figures/eval_residuals_timeseries_test_<version>.png`
  - `figures/eval_pred_timeseries_test_<version>.png`
- Feature importance CSV: `reports/feature_importance_<version>.csv`

## CRISP-DM Stage
Evaluation

In [ ]:
# Make the project root importable (so `import src...` works in notebooks)
import sys
from pathlib import Path

PROJECT_ROOT = Path("..").resolve()  # notebooks live in jupyter_notebooks/
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

print("Project root added to sys.path:", PROJECT_ROOT)

In [ ]:
import json
from dataclasses import asdict
from pathlib import Path

import joblib
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from src.config import DEFAULT_VERSION, get_paths
from src.features import load_features_latest
from src.modelling import FEATURE_COLS_NUM, FEATURE_COLS_CAT, TARGET_COL, time_split
from src.evaluation import regression_metrics, save_actual_vs_pred_plot

np.random.seed(42)

In [ ]:
VERSION = DEFAULT_VERSION
paths = get_paths(VERSION)

MODELS_DIR = paths.models_dir
REPORTS_DIR = paths.outputs_dir / "reports"
FIG_DIR = paths.outputs_dir / "figures"

REPORTS_DIR.mkdir(parents=True, exist_ok=True)
FIG_DIR.mkdir(parents=True, exist_ok=True)

print("Version:", VERSION)
print("Models dir:", MODELS_DIR)
print("Reports dir:", REPORTS_DIR)
print("Figures dir:", FIG_DIR)